In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import IRGenerator

printer = Post(PrettyPrint())

source = """
a=2 \n b = a + 3 \n H = %I %* %X \n c = true \n d = not c \n 
e = c and d \n f = c or d \n g = a <= b \n h = a >= b \n 
i = a == b \n k = a != b \n l = a - 1 \n m = a * b \n n = 2^3 \n
"""

source = """ 
a = 2 \n b = 5 \n 
if (a > 0) { \n b = 3 } \n
if (b < 0) { \n a = 5} \n
else { \n a = 10}
if (a < 0) { \n b = 10}
"""

source = """ 
n = 5 \n
while (n > 0) { \n n = n - 1}
"""

source = """
a = [2, 3, 4]
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
"""


source = """ 
a = sin(3.14 / 4)
b = abs(-5)
c = atan2(8, 5)
d = heaviside(2)
i = -4 + 4 * 1j
e = real(0 + 1j)
f = imag(1j)
g = conj(1 + 1j)
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
list = [1, 2, 3, 4]
b = list
initialize(q0)
// initialize(q1)
result = evolve(-(3.14 * #t / 4) %* %X , 1, r[0])
// result2 = evolve(-(3.14 / 4) %* %X , 1, q1)
result = evolve(-(3.14 / 4) %* (%X %* %X) , 1, r)
measurement = measure(q0)
"""

# source = """ 
# r = qreg(2)
# initialize(r)
# hamiltonian = -(3.14 * #t / 4) %* %X
# result = evolve(hamiltonian, 1, r[0])
# h2 = #s %* %X
# result2 = evolve(h2, 1, r[1])
# """

source = """ 
r = qreg(5)
q0 = r[0]
q1 = r[1]
q3 = r[3]
initialize(r)
s = [q0, q1]
// s = [q1, q0]
t = [q1, q3]
// result = evolve(%X , 1, q0)
result = evolve( %X %@ %X, 1, s)
result2 = evolve( %X %@ %X, 1, t)
//evolve(%X, 1, r[2])
"""

# source = "a = (#t + 4) * 2"

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
interpreter = IRGenerator(graph=cfg)
interpreter.run()
store = interpreter.status()
store
# store['q0'].state

dimensions are:
[[2, 2], [2, 2]]
Hamiltonian: 
Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
state before: 
Quantum object: dims=[[2, 2], [1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]
 [0.]
 [0.]]
dimensions are:
[[2, 2], [2, 2]]
Hamiltonian: 
Quantum object: dims=[[2, 2, 2], [2, 2, 2]], shape=(8, 8), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0.]]
state before: 
Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[0.54030206+0.j        ]
 [0.        +0.j        ]
 [0.        +0.j        ]
 [0.        +0.j        ]
 [0.        +0.j        ]
 [0.        +0.j        ]
 [0.        -0.84147114j]
 [0.    

{'r': [('r', 0), ('r', 1), ('r', 2), ('r', 3), ('r', 4)],
 'q0': ('r', 0),
 'q1': ('r', 1),
 'q3': ('r', 3),
 's': [('s', 0), ('s', 1)],
 ('s', 0): ('r', 0),
 ('s', 1): ('r', 1),
 't': [('t', 0), ('t', 1)],
 ('t', 0): ('r', 1),
 ('t', 1): ('r', 3),
 'result': array([0.54030206+0.j        , 0.        +0.j        ,
        0.        +0.j        , 0.        -0.84147114j]),
 'result2': array([ 0.29192632+0.j        ,  0.        +0.j        ,
         0.        +0.j        ,  0.        -0.45464859j,
         0.        +0.j        , -0.70807368+0.j        ,
         0.        -0.45464859j,  0.        +0.j        ])}

In [3]:
store = interpreter.status()
store

{'r': [('r', 0), ('r', 1), ('r', 2), ('r', 3), ('r', 4)],
 'q0': ('r', 0),
 'q1': ('r', 1),
 'q3': ('r', 3),
 's': [('s', 0), ('s', 1)],
 ('s', 0): ('r', 0),
 ('s', 1): ('r', 1),
 't': [('t', 0), ('t', 1)],
 ('t', 0): ('r', 1),
 ('t', 1): ('r', 3),
 'result': array([0.54030206+0.j        , 0.        +0.j        ,
        0.        +0.j        , 0.        -0.84147114j]),
 'result2': array([ 0.29192632+0.j        ,  0.        +0.j        ,
         0.        +0.j        ,  0.        -0.45464859j,
         0.        +0.j        , -0.70807368+0.j        ,
         0.        -0.45464859j,  0.        +0.j        ])}

In [4]:
# from oqd_core.compiler.analog.math.rules import SubstituteMathVar
# from oqd_core.interface.analog.expr import MathVar
# from oqd_compiler_infrastructure import Post
# substitute_pass = Post(SubstituteMathVar(MathVar(class_='MathVar', name='#s'), MathVar(class_='MathVar', name='#t') - 10))

# substitute_pass(store['a'])
registers = interpreter.interpreter.registers
registers

{('r', 0): <oqd_analog_emulator.interpreter.QubitRegister at 0x115223f90>,
 ('r', 1): <oqd_analog_emulator.interpreter.QubitRegister at 0x115223f90>,
 ('r', 2): <oqd_analog_emulator.interpreter.QubitObject at 0x11322c4d0>,
 ('r', 3): <oqd_analog_emulator.interpreter.QubitRegister at 0x115223f90>,
 ('r', 4): <oqd_analog_emulator.interpreter.QubitObject at 0x115872390>}

In [5]:
registers[('r', 0)].qubits

[('r', 0), ('r', 1), ('r', 3)]

In [6]:
registers[('r', 0)].state

Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[ 0.29192632+0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        -0.45464859j]
 [ 0.        +0.j        ]
 [-0.70807368+0.j        ]
 [ 0.        -0.45464859j]
 [ 0.        +0.j        ]]

In [7]:
registers[('r', 1)].state

Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[ 0.29192632+0.j        ]
 [ 0.        +0.j        ]
 [ 0.        +0.j        ]
 [ 0.        -0.45464859j]
 [ 0.        +0.j        ]
 [-0.70807368+0.j        ]
 [ 0.        -0.45464859j]
 [ 0.        +0.j        ]]

In [8]:
registers[('r', 0)].time

2.0

In [9]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'r',
   'value': {'class_': 'QuantumRegister', 'size': 5}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q0',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 0}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q1',
   'value': {'class_': 'Extract',
    'access': {'class_': 'Access', 'name': 'r'},
    'index': 1}},
  'preds': [2],
  'succs': [4],
  'exit_nodes': [],
  'edge_labels': {}},
 4: {'register_id': 4,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'q3',
   'value': {'cla